# Clustering Pola Penjualan Produk dengan FastDTW

Notebook ini merupakan alur analisis yang modular, reproducible, dan menjadi sumber analisis untuk dashboard pada tahap berikutnya. Seluruh fungsi/class analisis berada di folder `scripts/`; notebook hanya mengatur urutan analisis, menampilkan tabel/grafik, dan menjelaskan interpretasinya.

**Tujuan:** mengelompokkan produk berdasarkan kemiripan bentuk penjualan mingguan W0–W51 menggunakan enam metode clustering berbasis FastDTW.

**Prinsip utama:** fitur clustering berasal dari **data penjualan asli W0–W51** yang ditransformasikan menggunakan **Min-Max scaling per produk**. Nilai penjualan asli tetap digunakan untuk membaca volume, kontribusi, aktivitas, dan profil bisnis.

> Catatan ruang lingkup: notebook ini menyiapkan kebutuhan analisis terlebih dahulu. Folder `dashboard/` dan `app.py` belum dibuat atau diubah.

## 1. Persiapan proyek dan konfigurasi

Struktur proyek memisahkan data, notebook, script, model, dan output. Parameter yang mungkin berubah (rentang tuning, radius FastDTW, random seed, batas evaluasi, dan lokasi folder) dikelola melalui `config.py` agar tidak perlu mengedit banyak cell.

Jalankan notebook dari root proyek atau dari folder `notebook/`.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Cari root proyek dari working directory saat ini atau direktori induknya.
SEARCH_PATHS = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in SEARCH_PATHS if (path / "config.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT))

import config
from scripts.data import (
    load_sales_data,
    build_product_statistics,
    build_weekly_statistics,
)
from scripts.preprocessing import build_preprocessing_pipeline, transform_sales_data
from scripts.distance import fastdtw_numba, exact_dtw_distance
from scripts.clustering import METHODS
from scripts.experiment import (
    run_clustering_experiment,
    build_cluster_profiles,
    build_silhouette_detail,
    build_tuning_per_k,
    build_pca_coordinates,
)
from scripts.visualization import (
    build_dashboard_figures,
    create_cluster_pattern_figure,
    create_cluster_contribution_figure,
    create_weekly_sales_figure,
    create_top_products_figure,
    create_zero_weeks_figure,
    create_exploration_figures,
    create_metrics_comparison_figure,
    create_coverage_figure,
    create_similarity_heatmap,
    create_tuning_k_figure,
    create_dbscan_tuning_figure,
    create_silhouette_distribution_figure,
    create_silhouette_box_figure,
    create_pca_scatter_figure,
    create_weekly_diagnostics_figure,
)
from scripts.artifacts import save_experiment_artifacts, save_plotly_figures

for folder in [config.MODEL_DIR, config.OUTPUT_DIR, config.PLOT_DIR, config.DISTANCE_CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data:", config.DATA_FILE)
print("FastDTW radius:", config.FASTDTW_RADIUS)
print("Random state:", config.RANDOM_STATE)

Project root: c:\Users\Tsaqif\Downloads\Clustering ADL\coba\Ruang_Sales_Analysis Dashboard Styled
Data: C:\Users\Tsaqif\Downloads\Clustering ADL\coba\Ruang_Sales_Analysis Dashboard Styled\dataset\sales.csv
FastDTW radius: 2
Random state: 42


## 2. Memuat dan memvalidasi data

Dataset memiliki satu baris untuk satu produk, satu kolom ID (`Product_Code`), dan 52 kolom penjualan mingguan (`W0`–`W51`). Validasi memastikan ID unik, kolom lengkap, serta nilai penjualan numerik dan finite. Nilai kosong dan nilai negatif tidak diimputasi/dihapus secara diam-diam; keduanya akan menghasilkan error agar sumber data dapat diperiksa.

**Interpretasi:** seluruh 52 minggu digunakan sebagai satu seri untuk setiap produk. Minggu nol dipertahankan karena merupakan observasi penjualan aktual bernilai nol, bukan otomatis bukti kehabisan stok.

In [2]:
sales_df = load_sales_data(
    file_path=config.DATA_FILE,
    id_column=config.ID_COLUMN,
    week_columns=config.WEEK_COLUMNS,
    delimiter=",",
    allow_missing=False,
    allow_negative=False,
)

product_codes = sales_df[config.ID_COLUMN].astype(str).to_numpy()
W_original = sales_df[config.WEEK_COLUMNS].to_numpy(dtype=np.float64)
product_stats = build_product_statistics(sales_df, config.ID_COLUMN, config.WEEK_COLUMNS)
weekly_stats = build_weekly_statistics(sales_df, config.WEEK_COLUMNS)

quality_summary = pd.DataFrame({
    "Indikator": [
        "Jumlah produk", "Jumlah minggu", "Total pengamatan produk × minggu",
        "Jumlah Product_Code duplikat", "Jumlah nilai kosong", "Jumlah nilai negatif",
        "Jumlah observasi minggu nol", "Persentase observasi minggu nol",
        "Total volume penjualan", "Produk dengan ≥40 minggu nol",
    ],
    "Nilai": [
        len(sales_df), W_original.shape[1], W_original.size,
        int(sales_df[config.ID_COLUMN].duplicated().sum()),
        int(np.isnan(W_original).sum()), int((W_original < 0).sum()),
        int((W_original == 0).sum()), f"{100 * (W_original == 0).mean():.2f}%",
        f"{W_original.sum():,.0f}", int(((W_original == 0).sum(axis=1) >= 40).sum()),
    ],
})

display(quality_summary)
display(sales_df.head())
print("Dimensi data:", sales_df.shape)
print("Produk unik:", sales_df[config.ID_COLUMN].nunique())

,Indikator,Nilai
0,Jumlah produk,811
1,Jumlah minggu,52
2,Total pengamatan produk × minggu,42172
3,Jumlah Product_Code duplikat,0
4,Jumlah nilai kosong,0
5,Jumlah nilai negatif,0
6,Jumlah observasi minggu nol,11899
7,Persentase observasi minggu nol,28.22%
8,Total volume penjualan,"375,287"
9,Produk dengan ≥40 minggu nol,168


,Product_Code,W0,W1,W2,W3,W4,W5,W6,W7,W8,...,W42,W43,W44,W45,W46,W47,W48,W49,W50,W51
0,P1,11,12,10,8,13,12,14,21,6,...,4,7,8,10,12,3,7,6,5,10
1,P2,7,6,3,2,7,1,6,3,3,...,2,4,5,1,1,4,5,1,6,0
2,P3,7,11,8,9,10,8,7,13,12,...,6,14,5,5,7,8,14,8,8,7
3,P4,12,8,13,5,9,6,9,13,13,...,9,10,3,4,6,8,14,8,7,8
4,P5,8,5,13,11,6,7,9,14,9,...,7,11,7,12,6,6,5,11,8,9


Dimensi data: (811, 53)
Produk unik: 811


### Interpretasi kualitas data

Periksa ringkasan di atas sebelum melanjutkan. Jumlah produk unik harus sama dengan jumlah baris, jumlah minggu harus 52, serta jumlah nilai kosong dan negatif harus nol sesuai asumsi data penjualan. Produk tidak dibuang hanya karena memiliki banyak minggu nol; pola penjualan yang jarang tetap merupakan karakter data yang perlu dipertimbangkan pada segmentasi.

## 3. Ringkasan penjualan — halaman Ringkasan dashboard

Bagian ini memuat komponen utama ringkasan dashboard: tren total penjualan mingguan, 10 produk dengan volume tertinggi, dan distribusi jumlah minggu nol.

**Interpretasi yang perlu dibaca:** tren total dapat menunjukkan perubahan agregat, tetapi tidak menjelaskan produk penyebabnya. Produk dengan total tinggi dapat mendominasi volume agregat. Distribusi minggu nol membantu memisahkan produk yang rutin aktif dari produk yang penjualannya sporadis.

In [3]:
overview_figures = {
    "weekly_sales": create_weekly_sales_figure(weekly_stats, config),
    "top_products": create_top_products_figure(product_stats, 10, config),
    "zero_weeks": create_zero_weeks_figure(product_stats, config),
}
for figure_name, figure in overview_figures.items():
    print(figure_name)
    display(figure)

weekly_sales


top_products


zero_weeks


## 4. Eksplorasi data — halaman Eksplorasi dashboard

Eksplorasi mencakup distribusi total volume per produk, hubungan antara persentase minggu aktif dan total volume, tabel statistik produk, serta deret waktu satu produk. Filter/pencarian pada tabel tidak mengubah populasi yang digunakan untuk clustering.

**Interpretasi:** histogram total volume menunjukkan kemencengan dan kemungkinan produk ber-volume ekstrem. Scatter aktivitas–volume membantu melihat apakah volume tinggi berhubungan dengan frekuensi aktif. Keduanya merupakan analisis deskriptif pada data asli.

In [4]:
DEFAULT_PRODUCT = "P43" if "P43" in set(product_codes) else str(product_codes[0])
exploration_figures = create_exploration_figures(
    product_stats=product_stats,
    W_original=W_original,
    selected_product=DEFAULT_PRODUCT,
    config=config,
)
for figure_name, figure in exploration_figures.items():
    print(figure_name)
    display(figure)

display(product_stats.sort_values("Total volume", ascending=False).head(15))
print("Produk yang ditampilkan pada grafik detail:", DEFAULT_PRODUCT)

sales_distribution


activity_vs_volume


product_detail


,Produk,Total volume,Rata-rata / minggu,Minggu nol,Minggu aktif (%),Puncak mingguan,Minggu puncak
406,P409,2220.0,42.692308,0,100.0,73.0,W51
33,P34,1932.0,37.153846,0,100.0,55.0,W3
177,P178,1925.0,37.019231,0,100.0,53.0,W15
134,P135,1920.0,36.923077,0,100.0,56.0,W6
42,P43,1913.0,36.788462,0,100.0,61.0,W20
189,P190,1912.0,36.769231,0,100.0,54.0,W0
178,P179,1904.0,36.615385,0,100.0,55.0,W46
172,P173,1897.0,36.480769,0,100.0,54.0,W16
91,P92,1896.0,36.461538,0,100.0,59.0,W15
136,P137,1894.0,36.423077,0,100.0,54.0,W6


Produk yang ditampilkan pada grafik detail: P43


## 5. Preprocessing: Min-Max dari data penjualan asli

Tidak ada variabel `Normalized`, `min`, atau `max` yang digunakan sebagai sumber fitur. Langkah transformasi mengambil matriks penjualan asli `W_original`, lalu melakukan Min-Max per baris produk:

\[
X_{ij}^{*}=\frac{W_{ij}-\min(W_i)}{\max(W_i)-\min(W_i)}.
\]

Dengan demikian, setiap produk memiliki rentang 0–1. Jika seluruh penjualan suatu produk konstan, hasil transformasinya dibuat nol agar tidak terjadi pembagian nol. Scaling per produk dipilih karena tujuan utama clustering adalah membandingkan **bentuk pola**, bukan mengelompokkan produk semata-mata berdasarkan besarnya volume. Konsekuensinya, perbedaan skala penjualan antarproduk hilang pada input clustering. Oleh karena itu, semua interpretasi level/volume kembali menggunakan `W_original`.

> Catatan teknis: `sklearn.preprocessing.MinMaxScaler` secara default melakukan scaling per fitur/kolom. Untuk matriks dengan satu produk per baris, `RowWiseMinMaxScaler` menerapkan rumus Min-Max sepanjang 52 minggu pada setiap baris. Transformasi ini disimpan dalam objek `sklearn.pipeline.Pipeline`.

In [5]:
preprocessing_pipeline, X_scaled = transform_sales_data(
    W_original,
    feature_range=config.MINMAX_FEATURE_RANGE,
)

row_min = X_scaled.min(axis=1)
row_max = X_scaled.max(axis=1)
row_range = np.ptp(X_scaled, axis=1)

scaling_check = pd.DataFrame({
    "Pemeriksaan": [
        "Dimensi fitur clustering", "Minimum global hasil scaling", "Maksimum global hasil scaling",
        "Produk dengan minimum bukan 0", "Produk non-konstan dengan maksimum bukan 1",
        "Produk dengan seri konstan", "Sumber fitur",
    ],
    "Nilai": [
        str(X_scaled.shape), f"{X_scaled.min():.4f}", f"{X_scaled.max():.4f}",
        int(np.sum(np.abs(row_min) > 1e-10)),
        int(np.sum((np.ptp(W_original, axis=1) > 0) & (np.abs(row_max - 1) > 1e-10))),
        int(np.sum(np.ptp(W_original, axis=1) == 0)),
        "W0–W51 asli; bukan kolom normalisasi dari CSV",
    ],
})
display(scaling_check)
display(pd.DataFrame(X_scaled[:5], columns=config.WEEK_COLUMNS, index=product_codes[:5]))

,Pemeriksaan,Nilai
0,Dimensi fitur clustering,"(811, 52)"
1,Minimum global hasil scaling,0.0000
2,Maksimum global hasil scaling,1.0000
3,Produk dengan minimum bukan 0,0
4,Produk non-konstan dengan maksimum bukan 1,0
5,Produk dengan seri konstan,0
6,Sumber fitur,W0–W51 asli; bukan kolom normalisasi dari CSV


,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,...,W42,W43,W44,W45,W46,W47,W48,W49,W50,W51
P1,0.444444,0.500000,0.388889,0.277778,0.555556,0.500000,0.611111,1.000000,0.166667,0.611111,...,0.055556,0.222222,0.277778,0.388889,0.500000,0.000000,0.222222,0.166667,0.111111,0.388889
P2,0.700000,0.600000,0.300000,0.200000,0.700000,0.100000,0.600000,0.300000,0.300000,0.300000,...,0.200000,0.400000,0.500000,0.100000,0.100000,0.400000,0.500000,0.100000,0.600000,0.000000
P3,0.363636,0.727273,0.454545,0.545455,0.636364,0.454545,0.363636,0.909091,0.818182,0.272727,...,0.272727,1.000000,0.181818,0.181818,0.363636,0.454545,1.000000,0.454545,0.454545,0.363636
P4,0.588235,0.352941,0.647059,0.176471,0.411765,0.235294,0.411765,0.647059,0.647059,0.529412,...,0.411765,0.470588,0.058824,0.117647,0.235294,0.352941,0.705882,0.352941,0.294118,0.352941
P5,0.333333,0.133333,0.666667,0.533333,0.200000,0.266667,0.400000,0.733333,0.400000,0.400000,...,0.266667,0.533333,0.266667,0.600000,0.200000,0.200000,0.133333,0.533333,0.333333,0.400000


### Interpretasi hasil preprocessing

Setiap baris produk non-konstan seharusnya memiliki nilai minimum 0 dan maksimum 1. Penskalaan ini mempertahankan urutan relatif minggu dalam satu produk, tetapi menghilangkan informasi besaran volume absolut. Karena itu, jangan menyimpulkan bahwa dua produk dengan kurva Min-Max serupa memiliki tingkat penjualan atau kontribusi yang sama.

## 6. Pemeriksaan FastDTW

FastDTW mengukur kemiripan seri dengan memperbolehkan jalur alignment yang tidak harus memasangkan minggu ke minggu secara kaku. Implementasi yang digunakan memakai biaya lokal selisih kuadrat dan mengakarkan jumlah biaya sepanjang jalur. Radius mengatur lebar area pencarian pada pendekatan multiresolusi; radius bukan berarti maksimum pergeseran minggu.

Sebelum menghitung matriks penuh, satu pasang seri diperiksa: jarak, panjang alignment path, dan konsistensi biaya yang dihitung dari path.

In [6]:
sample_a = X_scaled[0]
sample_b = X_scaled[1]
fast_distance, path_i, path_j = fastdtw_numba(
    sample_a, sample_b, radius=config.FASTDTW_RADIUS, return_path=True
)
exact_distance = exact_dtw_distance(sample_a, sample_b)
path_distance = float(np.sqrt(np.sum((sample_a[path_i] - sample_b[path_j]) ** 2)))

distance_check = pd.DataFrame({
    "Pemeriksaan": ["FastDTW", "DTW eksak", "Jarak dihitung ulang dari path", "Jumlah pasangan alignment"],
    "Nilai": [fast_distance, exact_distance, path_distance, len(path_i)],
})
display(distance_check)
assert np.isfinite(fast_distance) and len(path_i) == len(path_j)
assert np.isclose(fast_distance, path_distance, rtol=1e-8, atol=1e-8)
print("Pemeriksaan path FastDTW lulus.")

,Pemeriksaan,Nilai
0,FastDTW,1.284331
1,DTW eksak,1.237530
2,Jarak dihitung ulang dari path,1.284331
3,Jumlah pasangan alignment,69.000000


Pemeriksaan path FastDTW lulus.


## 7. Pemodelan enam metode clustering

Satu matriks jarak FastDTW dihitung dan digunakan bersama oleh seluruh metode agar perhitungan tidak diulang untuk setiap kandidat. Cache matriks diberi identitas berdasarkan hash data hasil Min-Max dan radius FastDTW.

Metode yang dituning:

1. **K-Means + DBA** — pusat cluster berupa barycenter time series yang diperbarui melalui alignment.
2. **K-Medoids** — pusat cluster merupakan produk aktual yang meminimalkan total jarak dalam cluster.
3. **Agglomerative** — hierarchical clustering dengan linkage yang ditentukan.
4. **DBSCAN** — membentuk kelompok padat dan dapat menandai observasi sebagai noise (`-1`).
5. **Fuzzy C-Means + DBA** — setiap produk memiliki derajat keanggotaan fuzzy dan pusat berupa barycenter.
6. **Fuzzy C-Medoids** — pusat berupa medoid dengan keanggotaan fuzzy.

Seluruh metode menerima input mentah `W_original` melalui pipeline. Langkah `minmax` dijalankan terlebih dahulu, kemudian estimator clustering bekerja pada fitur hasil scaling dan matriks jarak FastDTW. Kandidat, seed, serta iterasi diatur dalam `config.py`.

In [7]:
experiment = run_clustering_experiment(
    W_original=W_original,
    product_codes=product_codes,
    config=config,
)

X_scaled = experiment["X_scaled"]
D = experiment["D"]
pipelines = experiment["pipelines"]
metrics = experiment["metrics"]
labels_df = experiment["labels"]

print("Matriks jarak:", D.shape)
print("Cache ditemukan:", experiment["distance_cache_hit"])
print("Cache file:", experiment["distance_cache_path"])
print("Pemimpin skor multi-indeks:", experiment["best_method"])
print("Status seleksi:", experiment["selection_status"])
for warning in experiment["selection_warnings"]:
    print("PERHATIAN:", warning)
display(pd.DataFrame([
    {"Metode": method, "Parameter terpilih": str(experiment["best_parameters"][method])}
    for method in METHODS
]))

[KMeans_DBA] 1 kandidat...
[KMeans_DBA] selesai dalam 2.2 detik.
[KMedoids] 1 kandidat...
[KMedoids] selesai dalam 0.1 detik.
[Agglomerative] 2 kandidat...
[Agglomerative] selesai dalam 0.1 detik.
[DBSCAN] 28 kandidat...
[DBSCAN] selesai dalam 1.4 detik.
[FuzzyCMeans] 2 kandidat...
[FuzzyCMeans] selesai dalam 2.2 detik.
[FuzzyCMedoids] 2 kandidat...
[FuzzyCMedoids] selesai dalam 0.2 detik.
[Stabilitas] KMeans_DBA...
[Stabilitas] KMedoids...
[Stabilitas] Agglomerative...
[Stabilitas] DBSCAN...
[Stabilitas] FuzzyCMeans...
[Stabilitas] FuzzyCMedoids...
Matriks jarak: (811, 811)
Cache ditemukan: True
Cache file: C:\Users\Tsaqif\Downloads\Clustering ADL\coba\Ruang_Sales_Analysis Dashboard Styled\output\distance_cache\fastdtw_2c6031e3e918381b_r2.npy
Pemimpin skor multi-indeks: Agglomerative
Status seleksi: PASS_SHAPE_GUARDRAIL
PERHATIAN: Lolos guardrail eta-squared bukan bukti validitas eksternal; interpretasi bisnis tetap perlu divalidasi.


,Metode,Parameter terpilih
0,KMeans_DBA,{'n_clusters': 2}
1,KMedoids,{'n_clusters': 2}
2,Agglomerative,"{'n_clusters': 2, 'linkage_method': 'average'}"
3,DBSCAN,"{'min_samples': 7, 'eps': 0.9770108928115545, ..."
4,FuzzyCMeans,"{'n_clusters': 2, 'fuzziness': 2.0}"
5,FuzzyCMedoids,"{'n_clusters': 2, 'fuzziness': 2.0}"


## 8. Evaluasi dan perbandingan metode — halaman Perbandingan dashboard

Metrik utama harus dibaca secara bersamaan:

- **Silhouette FastDTW**: semakin tinggi, semakin baik pemisahan relatif berdasarkan jarak yang digunakan untuk clustering.
- **DBI medoid FastDTW**: semakin rendah, semakin baik; ini adaptasi DBI dengan medoid dan jarak FastDTW, bukan DBI standar.
- **DBI Euclidean dan CHI Euclidean**: dihitung pada fitur Min-Max untuk diagnostik tambahan. Keduanya memakai geometri Euclidean, bukan geometri FastDTW.
- **Coverage**: proporsi produk yang mendapat label cluster. Untuk DBSCAN, noise tidak dihitung sebagai anggota cluster.
- **ARI/NMI antar-metode**: kesepakatan antara dua partisi; bukan akurasi terhadap label benar.
- **Stabilitas ARI/NMI**: konsistensi hasil beberapa seed untuk metode yang memiliki inisialisasi acak. Metode deterministik dicatat sebagai tidak memerlukan uji seed.

Skor multi-indeks hanya heuristik penyaring. Tidak tersedia ground truth sehingga tidak ada metrik akurasi klasifikasi.

In [8]:
comparison_columns = [
    "Metode", "Jumlah_Cluster", "Jumlah_Noise", "Coverage", "Anggota_Min",
    "Silhouette_FastDTW", "DBI_Medoid_FastDTW", "DBI_MinMax_Euclidean",
    "CHI_MinMax_Euclidean", "Stabilitas_ARI", "Stabilitas_NMI",
    "Eta2_Minggu_Nol", "Skor_Multi_Indeks", "Layak_Semua_Produk",
]
display(metrics[comparison_columns].round(4))

tuning_per_k = build_tuning_per_k(experiment["tuning"])
ari_matrix = experiment["ari"]
nmi_matrix = experiment["nmi"]
display(experiment["stability"].round(4))
display(ari_matrix.round(3))
display(nmi_matrix.round(3))

,Metode,Jumlah_Cluster,Jumlah_Noise,Coverage,Anggota_Min,Silhouette_FastDTW,DBI_Medoid_FastDTW,DBI_MinMax_Euclidean,CHI_MinMax_Euclidean,Stabilitas_ARI,Stabilitas_NMI,Eta2_Minggu_Nol,Skor_Multi_Indeks,Layak_Semua_Produk
0,Agglomerative,2,0,1.0000,11,0.2484,1.1473,4.5002,2.8843,NaN,NaN,0.0190,1.0,True
1,KMeans_DBA,2,0,1.0000,250,0.3311,1.3105,1.6837,240.0774,0.8298,0.7636,0.8153,NaN,True
2,KMedoids,2,0,1.0000,289,0.3334,1.2957,1.6224,280.4141,0.9576,0.9172,0.8674,NaN,True
3,DBSCAN,2,272,0.6646,11,0.2300,0.6369,2.7921,6.2203,NaN,NaN,0.0412,NaN,False
4,FuzzyCMeans,2,0,1.0000,330,0.2111,1.6357,2.2428,151.1000,0.5875,0.4895,0.3984,NaN,True
5,FuzzyCMedoids,2,0,1.0000,393,0.2128,1.2219,1.9753,198.7854,0.5852,0.5764,0.4162,NaN,True


,Metode,Stabilitas_ARI,Stabilitas_NMI,Status
0,KMeans_DBA,0.8298,0.7636,Uji 3 seed
1,KMedoids,0.9576,0.9172,Uji 3 seed
2,Agglomerative,NaN,NaN,Deterministik; uji seed tidak diperlukan
3,DBSCAN,NaN,NaN,Deterministik; uji seed tidak diperlukan
4,FuzzyCMeans,0.5875,0.4895,Uji 3 seed
5,FuzzyCMedoids,0.5852,0.5764,Uji 3 seed


,KMeans_DBA,KMedoids,Agglomerative,DBSCAN,FuzzyCMeans,FuzzyCMedoids
KMeans_DBA,1.000,0.788,0.034,-0.001,0.387,0.287
KMedoids,0.788,1.000,0.022,0.001,0.417,0.375
Agglomerative,0.034,0.022,1.000,0.024,0.013,-0.001
DBSCAN,-0.001,0.001,0.024,1.000,0.002,0.018
FuzzyCMeans,0.387,0.417,0.013,0.002,1.000,0.428
FuzzyCMedoids,0.287,0.375,-0.001,0.018,0.428,1.000


,KMeans_DBA,KMedoids,Agglomerative,DBSCAN,FuzzyCMeans,FuzzyCMedoids
KMeans_DBA,1.000,0.705,0.047,0.040,0.303,0.319
KMedoids,0.705,1.000,0.039,0.030,0.316,0.366
Agglomerative,0.047,0.039,1.000,0.039,0.033,0.013
DBSCAN,0.040,0.030,0.039,1.000,0.024,0.028
FuzzyCMeans,0.303,0.316,0.033,0.024,1.000,0.374
FuzzyCMedoids,0.319,0.366,0.013,0.028,0.374,1.000


### Interpretasi evaluasi

Nilai Silhouette yang lebih tinggi menunjukkan cluster lebih terpisah pada jarak FastDTW, tetapi belum tentu berarti segmentasi paling berguna secara bisnis. Skor perlu dibaca bersama jumlah anggota terkecil, coverage, DBI, stabilitas, serta profil penjualan asli. Khusus DBSCAN, Silhouette hanya dihitung pada produk non-noise sehingga tidak boleh dibandingkan tanpa melihat coverage. ARI/NMI yang tinggi antarmetode berarti partisi lebih mirip, bukan lebih benar.

**Pemeriksaan dominasi minggu nol:** `Eta2_Minggu_Nol` mengukur proporsi variasi jumlah minggu tanpa penjualan yang dapat diterangkan oleh label cluster. Nilai mendekati 1 menunjukkan partisi sangat berkaitan dengan frekuensi minggu nol. Ini penting karena Min-Max per produk tidak menghapus pola nol; model dapat lebih banyak memisahkan produk berdasarkan sparsity daripada bentuk tren aktifnya. Jika nilai ini melewati ambang `SHAPE_MAX_ETA_ZERO`, hasil tidak lolos guardrail shape dan ranking hanya ditampilkan sebagai fallback.

In [9]:
comparison_figures = {
    "metrics_comparison": create_metrics_comparison_figure(metrics, config),
    "coverage": create_coverage_figure(metrics, config),
    "ari_heatmap": create_similarity_heatmap(ari_matrix, "ARI", config),
    "nmi_heatmap": create_similarity_heatmap(nmi_matrix, "NMI", config),
    "tuning_k": create_tuning_k_figure(tuning_per_k, config),
    "dbscan_tuning": create_dbscan_tuning_figure(experiment["tuning"]["DBSCAN"], config),
}
for figure_name, figure in comparison_figures.items():
    print(figure_name)
    display(figure)

metrics_comparison


coverage


ari_heatmap


nmi_heatmap


tuning_k


dbscan_tuning


### Membaca hasil tuning

Grafik tuning memperlihatkan perubahan pemisahan ketika jumlah cluster berubah. Untuk Agglomerative, jumlah cluster yang dilaporkan merupakan jumlah cluster **aktual** dari hasil pemotongan dendrogram; jumlah aktual dapat berbeda dari target pada kondisi tertentu. DBSCAN tidak mempunyai parameter `k`; evaluasinya dilakukan terhadap kombinasi `eps` dan `min_samples`, dengan perhatian khusus pada proporsi noise.

## 9. Profil dan segmentasi produk — halaman Segmentasi dashboard

Profil setiap cluster dihitung kembali dari penjualan **asli**. Setiap cluster diringkas berdasarkan jumlah produk, porsi volume, median total penjualan per produk, minggu nol, persentase minggu aktif, perbedaan rerata dua paruh periode, dan medoid.

Fuzzy C-Means dipakai sebagai tampilan awal mengikuti pilihan dashboard yang diberikan. Ini hanya pilihan tampilan awal, bukan rekomendasi metode. Pemimpin ranking multi-indeks dihitung terpisah dan dapat berstatus fallback jika tidak ada metode yang memenuhi guardrail asosiasi jumlah minggu nol.

In [10]:
profiles_by_method = {
    method: build_cluster_profiles(
        W_original=W_original,
        product_codes=product_codes,
        labels=pipelines[method].named_steps["cluster"].labels_,
        distance_matrix=D,
        method=method,
    )
    for method in METHODS
}

display_method = config.DEFAULT_METHOD if config.DEFAULT_METHOD in pipelines else experiment["best_method"]
display_labels = pipelines[display_method].named_steps["cluster"].labels_
display_profiles = profiles_by_method[display_method]

print("Metode pada tampilan awal:", display_method)
print("Pemimpin skor multi-indeks (sementara):", experiment["best_method"])
print("Status seleksi:", experiment["selection_status"])
display(display_profiles.drop(columns="Medoid_Index").round(2))

Metode pada tampilan awal: KMedoids
Pemimpin skor multi-indeks (sementara): Agglomerative
Status seleksi: PASS_SHAPE_GUARDRAIL


,Metode,Cluster,Jumlah_Produk,Persen_Produk,Total_Volume,Porsi_Volume,Median_Total_Sales,Rata2_Total_Sales,Median_Minggu_Nol,Rata2_Minggu_Nol,Persen_Minggu_Aktif,Rerata_Penjualan_Paruh_1,Rerata_Penjualan_Paruh_2,Perubahan_Paruh_2_vs_1_Persen,Produk_Medoid
0,KMedoids,0,522,64.36,368945.0,98.31,475.0,706.79,0.0,1.40,97.30,14.86,12.33,-17.01,P200
1,KMedoids,1,289,35.64,6342.0,1.69,13.0,21.94,42.0,38.64,25.69,0.35,0.49,40.41,P731


### Profil semua metode

Tabel berikut diperlukan agar pemilihan metode di dashboard nantinya hanya mengganti sumber label dan profil, bukan menghitung ulang analisis. Perhatikan bahwa nomor cluster bersifat lokal untuk tiap metode: C0 pada K-Medoids tidak otomatis berisi produk yang sama dengan C0 pada Fuzzy C-Means.

In [11]:
for method in METHODS:
    print(f"Profil cluster — {method}")
    display(profiles_by_method[method].drop(columns="Medoid_Index").round(2))

Profil cluster — KMeans_DBA


,Metode,Cluster,Jumlah_Produk,Persen_Produk,Total_Volume,Porsi_Volume,Median_Total_Sales,Rata2_Total_Sales,Median_Minggu_Nol,Rata2_Minggu_Nol,Persen_Minggu_Aktif,Rerata_Penjualan_Paruh_1,Rerata_Penjualan_Paruh_2,Perubahan_Paruh_2_vs_1_Persen,Produk_Medoid
0,KMeans_DBA,0,561,69.17,370451.0,98.71,463.0,660.34,0.0,3.13,93.98,13.86,11.54,-16.76,P200
1,KMeans_DBA,1,250,30.83,4836.0,1.29,11.0,19.34,43.0,40.57,21.98,0.32,0.42,30.95,P731


Profil cluster — KMedoids


,Metode,Cluster,Jumlah_Produk,Persen_Produk,Total_Volume,Porsi_Volume,Median_Total_Sales,Rata2_Total_Sales,Median_Minggu_Nol,Rata2_Minggu_Nol,Persen_Minggu_Aktif,Rerata_Penjualan_Paruh_1,Rerata_Penjualan_Paruh_2,Perubahan_Paruh_2_vs_1_Persen,Produk_Medoid
0,KMedoids,0,522,64.36,368945.0,98.31,475.0,706.79,0.0,1.40,97.30,14.86,12.33,-17.01,P200
1,KMedoids,1,289,35.64,6342.0,1.69,13.0,21.94,42.0,38.64,25.69,0.35,0.49,40.41,P731


Profil cluster — Agglomerative


,Metode,Cluster,Jumlah_Produk,Persen_Produk,Total_Volume,Porsi_Volume,Median_Total_Sales,Rata2_Total_Sales,Median_Minggu_Nol,Rata2_Minggu_Nol,Persen_Minggu_Aktif,Rerata_Penjualan_Paruh_1,Rerata_Penjualan_Paruh_2,Perubahan_Paruh_2_vs_1_Persen,Produk_Medoid
0,Agglomerative,0,11,1.36,200.0,0.05,13.0,18.18,39.0,37.18,28.50,0.30,0.40,29.89,P717
1,Agglomerative,1,800,98.64,375087.0,99.95,202.5,468.86,1.0,14.36,72.38,9.82,8.22,-16.29,P436


... (2 outputs left out)

Profil cluster — FuzzyCMeans


,Metode,Cluster,Jumlah_Produk,Persen_Produk,Total_Volume,Porsi_Volume,Median_Total_Sales,Rata2_Total_Sales,Median_Minggu_Nol,Rata2_Minggu_Nol,Persen_Minggu_Aktif,Rerata_Penjualan_Paruh_1,Rerata_Penjualan_Paruh_2,Perubahan_Paruh_2_vs_1_Persen,Produk_Medoid
0,FuzzyCMeans,0,481,59.31,332087.0,88.49,472.0,690.41,0.0,4.66,91.04,14.53,12.02,-17.29,P548
1,FuzzyCMeans,1,330,40.69,43200.0,11.51,21.0,130.91,37.0,29.26,43.72,2.62,2.41,-8.05,P731


Profil cluster — FuzzyCMedoids


,Metode,Cluster,Jumlah_Produk,Persen_Produk,Total_Volume,Porsi_Volume,Median_Total_Sales,Rata2_Total_Sales,Median_Minggu_Nol,Rata2_Minggu_Nol,Persen_Minggu_Aktif,Rerata_Penjualan_Paruh_1,Rerata_Penjualan_Paruh_2,Perubahan_Paruh_2_vs_1_Persen,Produk_Medoid
0,FuzzyCMedoids,0,393,48.46,317442.0,84.59,500.0,807.74,0.0,1.93,96.29,17.06,14.00,-17.92,P548
1,FuzzyCMedoids,1,418,51.54,57845.0,15.41,23.5,138.39,34.0,26.65,48.75,2.75,2.57,-6.64,P794


### Visualisasi cluster: penjualan asli vs pola Min-Max

Grafik kiri menunjukkan median penjualan asli per minggu. Grafik kanan menunjukkan median pola yang sudah di-Min-Max per produk. Pita P10–P90 menunjukkan variasi anggota di sekitar median. Kedua tampilan menjawab pertanyaan berbeda: pola ternormalisasi digunakan untuk memahami bentuk yang mendasari clustering, sedangkan penjualan asli dipakai untuk memahami skala bisnis.

Grafik kontribusi membandingkan porsi jumlah produk dengan porsi total volume. Jika porsi volume lebih besar daripada porsi produk, cluster tersebut menyumbang volume rata-rata yang relatif lebih besar per produk.

In [12]:
segment_figures_by_method = {}
for method in METHODS:
    method_labels = pipelines[method].named_steps["cluster"].labels_
    segment_figures_by_method[f"segment_{method}_original"] = create_cluster_pattern_figure(
        W_original, method_labels, "Median volume · penjualan asli", config
    )
    segment_figures_by_method[f"segment_{method}_minmax"] = create_cluster_pattern_figure(
        X_scaled, method_labels, "Median nilai Min-Max · pola ternormalisasi", config
    )
    segment_figures_by_method[f"segment_{method}_contribution"] = create_cluster_contribution_figure(
        profiles_by_method[method], config
    )

for figure_name in [
    f"segment_{display_method}_original",
    f"segment_{display_method}_minmax",
    f"segment_{display_method}_contribution",
]:
    print(figure_name)
    display(segment_figures_by_method[figure_name])

segment_KMedoids_original


segment_KMedoids_minmax


segment_KMedoids_contribution


### Menelusuri anggota cluster dan produk representatif

Medoid adalah produk anggota yang meminimalkan jumlah jarak FastDTW ke seluruh anggota lain pada cluster yang sama. Medoid bukan produk dengan volume terbesar atau terkecil; ia merupakan produk yang paling representatif menurut ukuran jarak dalam cluster. Tabel anggota dapat difilter/pencarian pada tahap dashboard tanpa mengubah label clustering.

In [13]:
selected_cluster = int(sorted(np.unique(display_labels))[0])
selected_mask = display_labels == selected_cluster
selected_members = product_stats.loc[selected_mask].copy().sort_values("Total volume", ascending=False)
selected_profile = display_profiles.loc[display_profiles["Cluster"] == selected_cluster].iloc[0]

print("Cluster yang ditelusuri:", selected_cluster)
print("Jumlah anggota:", int(selected_mask.sum()))
print("Produk medoid:", selected_profile["Produk_Medoid"])
print("Median total penjualan per produk:", f"{selected_profile['Median_Total_Sales']:,.0f}")
print("Median jumlah minggu nol:", selected_profile["Median_Minggu_Nol"])
print("Porsi volume:", f"{selected_profile['Porsi_Volume']:.2f}%")
display(selected_members.head(30))

Cluster yang ditelusuri: 0
Jumlah anggota: 522
Produk medoid: P200
Median total penjualan per produk: 475
Median jumlah minggu nol: 0.0
Porsi volume: 98.31%


,Produk,Total volume,Rata-rata / minggu,Minggu nol,Minggu aktif (%),Puncak mingguan,Minggu puncak
406,P409,2220.0,42.692308,0,100.0,73.0,W51
33,P34,1932.0,37.153846,0,100.0,55.0,W3
177,P178,1925.0,37.019231,0,100.0,53.0,W15
134,P135,1920.0,36.923077,0,100.0,56.0,W6
42,P43,1913.0,36.788462,0,100.0,61.0,W20
189,P190,1912.0,36.769231,0,100.0,54.0,W0
178,P179,1904.0,36.615385,0,100.0,55.0,W46
172,P173,1897.0,36.480769,0,100.0,54.0,W16
91,P92,1896.0,36.461538,0,100.0,59.0,W15
136,P137,1894.0,36.423077,0,100.0,54.0,W6


### Derajat keanggotaan fuzzy

Untuk Fuzzy C-Means dan Fuzzy C-Medoids, nilai keanggotaan menunjukkan derajat kedekatan relatif produk pada setiap cluster, bukan probabilitas bahwa label benar dan bukan akurasi. Nilai keanggotaan yang mirip antarcluster menandakan produk tersebut berada di area peralihan.

In [14]:
fuzzy_method = display_method if display_method in {"FuzzyCMeans", "FuzzyCMedoids"} else "FuzzyCMeans"
fuzzy_clusterer = pipelines[fuzzy_method].named_steps["cluster"]
if fuzzy_clusterer.membership_ is not None:
    membership_table = pd.DataFrame(
        fuzzy_clusterer.membership_,
        columns=[f"Keanggotaan_C{i}" for i in range(fuzzy_clusterer.membership_.shape[1])],
    )
    membership_table.insert(0, "Product_Code", product_codes)
    membership_table["Cluster"] = fuzzy_clusterer.labels_
    membership_table["Confidence_Maksimum"] = fuzzy_clusterer.membership_.max(axis=1)
    display(membership_table.sort_values("Confidence_Maksimum").head(20))
    print("Rata-rata confidence maksimum:", round(float(membership_table["Confidence_Maksimum"].mean()), 4))
else:
    print("Keanggotaan fuzzy tidak tersedia pada model yang ditampilkan.")

,Product_Code,Keanggotaan_C0,Keanggotaan_C1,Cluster,Confidence_Maksimum
201,P202,0.500037,0.499963,0,0.500037
268,P269,0.500174,0.499826,0,0.500174
370,P372,0.500252,0.499748,0,0.500252
529,P532,0.499708,0.500292,1,0.500292
400,P403,0.499653,0.500347,1,0.500347
553,P556,0.499213,0.500787,1,0.500787
610,P613,0.500990,0.499010,0,0.500990
781,P790,0.498957,0.501043,1,0.501043
795,P804,0.501386,0.498614,0,0.501386
158,P159,0.501610,0.498390,0,0.501610


Rata-rata confidence maksimum: 0.5829


## 10. Diagnostik kualitas cluster

Silhouette individual digunakan untuk memeriksa apakah produk tertentu lebih dekat ke cluster lain. Nilai negatif berarti rata-rata jarak produk ke cluster lain lebih kecil daripada jarak ke cluster yang ditempatinya. Periksa persentase anggota bersilhouette negatif per cluster, bukan hanya rata-rata keseluruhan.

In [15]:
silhouette_detail, silhouette_summary = build_silhouette_detail(display_labels, D)
if not silhouette_detail.empty:
    silhouette_detail.insert(1, "Product_Code", product_codes[silhouette_detail["Index"].to_numpy()])
    display(silhouette_summary.round(4))
    display(create_silhouette_distribution_figure(silhouette_detail, config))
    display(create_silhouette_box_figure(silhouette_detail, config))
    print("Persentase silhouette negatif:", f"{100 * (silhouette_detail['Silhouette'] < 0).mean():.2f}%")
else:
    print("Silhouette individual tidak dapat dihitung: jumlah cluster/anggota tidak memenuhi syarat.")

,Cluster,Jumlah_Produk,Rata_Silhouette,Median_Silhouette,Persen_Silhouette_Negatif
0,0,522,0.3811,0.4077,0.0000
1,1,289,0.2471,0.2732,7.2664


Persentase silhouette negatif: 2.59%


### PCA untuk eksplorasi visual

PCA memproyeksikan fitur Min-Max 52 minggu menjadi dua komponen agar struktur kelompok dapat dilihat pada bidang dua dimensi. PCA di sini **hanya visualisasi**. Label tetap dihasilkan dari clustering berbasis FastDTW, bukan dari jarak pada koordinat PCA.

In [16]:
pca_data = build_pca_coordinates(X_scaled, display_labels, config.RANDOM_STATE)
display(create_pca_scatter_figure(pca_data, config))

## 11. Audit mingguan dan perubahan aktivitas

Grafik ini merupakan pemeriksaan deskriptif tambahan dari halaman ringkasan/metodologi. Total penjualan dan persentase produk aktif ditampilkan terpisah untuk menghindari penafsiran perubahan volume total sebagai perubahan aktivitas yang sama.

In [17]:
display(create_weekly_diagnostics_figure(weekly_stats, config))

weekly_comparison = weekly_stats.loc[weekly_stats["Minggu"].between(22, 28)].copy()
display(weekly_comparison.round(2))
week_24 = float(weekly_stats.loc[weekly_stats["Minggu"] == 24, "Total_Penjualan"].iloc[0])
week_26 = float(weekly_stats.loc[weekly_stats["Minggu"] == 26, "Total_Penjualan"].iloc[0])
change_24_26 = 100.0 * (week_26 / week_24 - 1.0) if week_24 else np.nan
print("Perubahan total W24 ke W26 (%):", round(change_24_26, 2))
print("Produk dengan penjualan turun W24 → W26:", f"{100 * (W_original[:, 26] < W_original[:, 24]).mean():.1f}%")
print("Produk dengan penjualan tetap W24 → W26:", f"{100 * (W_original[:, 26] == W_original[:, 24]).mean():.1f}%")
print("Produk dengan penjualan naik W24 → W26:", f"{100 * (W_original[:, 26] > W_original[:, 24]).mean():.1f}%")

,Minggu,Total_Penjualan,Median_Penjualan_Produk,Persen_Produk_Aktif
22,22,8031.0,4.0,72.75
23,23,7998.0,4.0,72.87
24,24,8245.0,5.0,77.56
25,25,7212.0,5.0,78.91
26,26,5637.0,3.0,69.67
27,27,5834.0,3.0,69.17
28,28,5988.0,3.0,69.42


Perubahan total W24 ke W26 (%): -31.63
Produk dengan penjualan turun W24 → W26: 59.4%
Produk dengan penjualan tetap W24 → W26: 25.8%
Produk dengan penjualan naik W24 → W26: 14.8%


## 12. Menyimpan model, pipeline, tabel, dan visualisasi

Seluruh keluaran berasal dari objek eksperimen yang sama. Label, metrik, profil, dan grafik tidak digabung dari run atau preprocessing yang berbeda.

Pipeline yang disimpan menerima data mentah berformat produk × 52 minggu. Saat `predict()` dipanggil melalui pipeline, data baru terlebih dahulu menjalani Min-Max per produk, lalu ditempatkan ke prototype cluster menurut FastDTW. Untuk DBSCAN, prediksi menggunakan sampel inti dan dapat menghasilkan `-1` bila tidak ada core sample dalam radius `eps`.

In [18]:
# Kumpulan figure yang sama dapat dipanggil oleh notebook dan dashboard nantinya.
figures_for_default_method = build_dashboard_figures(
    W_original=W_original,
    product_codes=product_codes,
    product_stats=product_stats,
    weekly_stats=weekly_stats,
    X_scaled=X_scaled,
    labels=display_labels,
    profiles=display_profiles,
    metrics=metrics,
    ari=ari_matrix,
    nmi=nmi_matrix,
    tuning_per_k=tuning_per_k,
    tuning_dbscan=experiment["tuning"]["DBSCAN"],
    silhouette_detail=silhouette_detail,
    pca_data=pca_data,
    selected_product=DEFAULT_PRODUCT,
    config=config,
)

all_plotly_figures = {**figures_for_default_method, **segment_figures_by_method}
saved_figure_paths = save_plotly_figures(all_plotly_figures, config.PLOT_DIR)

# Profil silhouette dan PCA mengikuti metode tampilan default yang terpilih.
artifact_paths = save_experiment_artifacts(
    experiment=experiment,
    data=sales_df,
    product_stats=product_stats,
    weekly_stats=weekly_stats,
    profiles_by_method=profiles_by_method,
    silhouette_detail=silhouette_detail,
    silhouette_summary=silhouette_summary,
    pca_data=pca_data,
    config=config,
)

print("Pipeline tersimpan:", artifact_paths["pipelines"])
print("Preprocessing pipeline:", artifact_paths["preprocessing_pipeline"])
print("Jumlah tabel/metadata tersimpan:", len(artifact_paths))
print("Jumlah visualisasi Plotly HTML:", len(saved_figure_paths))
display(pd.DataFrame({"Artefak": list(artifact_paths), "Path": [str(path) for path in artifact_paths.values()]}))

Pipeline tersimpan: C:\Users\Tsaqif\Downloads\Clustering ADL\coba\Ruang_Sales_Analysis Dashboard Styled\model\clustering_pipelines.joblib
Preprocessing pipeline: C:\Users\Tsaqif\Downloads\Clustering ADL\coba\Ruang_Sales_Analysis Dashboard Styled\model\preprocessing_pipeline.joblib
Jumlah tabel/metadata tersimpan: 27
Jumlah visualisasi Plotly HTML: 37


,Artefak,Path
0,product_labels,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
1,metrics_comparison,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
2,stability,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
3,ari_matrix,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
4,nmi_matrix,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
5,product_statistics,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
6,weekly_statistics,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
7,silhouette_per_product,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
8,silhouette_per_cluster,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...
9,pca_coordinates,C:\Users\Tsaqif\Downloads\Clustering ADL\coba\...


## 13. Kesimpulan dan batas interpretasi

Metode dengan skor multi-indeks tertinggi **tidak otomatis menjadi rekomendasi final**. Pertama, metode harus memenuhi coverage penuh, ukuran cluster minimum, dan guardrail asosiasi jumlah minggu nol. Bila tidak ada metode yang lolos guardrail shape, notebook tetap menampilkan pemimpin skor multi-indeks sebagai **hasil sementara/fallback**, disertai peringatan kualitas. Silhouette, stabilitas, dan profil bisnis tetap harus diperiksa bersama.

Hasil clustering bersifat deskriptif/eksploratif. Data tidak menyediakan label ground truth, tanggal kalender, harga, margin, stok, promosi, maupun identitas pelanggan. Karena itu, cluster tidak boleh ditafsirkan sebagai kelas benar, prediksi, ataupun keputusan persediaan otomatis.

In [19]:
best_row = metrics.set_index("Metode").loc[experiment["best_method"]]
display_row = metrics.set_index("Metode").loc[display_method]

selection_label = (
    "Pemimpin skor multi-indeks (sementara/fallback)"
    if experiment["selection_status"] == "FALLBACK_NO_SHAPE_GUARDRAIL"
    else "Pemimpin ranking setelah guardrail"
)
warning_lines = "\n".join(f"- {warning}" for warning in experiment["selection_warnings"])

conclusion_text = f"""
### Ringkasan hasil eksperimen

- **Jumlah data:** {len(sales_df):,} produk dan {W_original.shape[1]} minggu.
- **Preprocessing clustering:** Min-Max per produk dari nilai asli W0–W51.
- **Jarak pola:** FastDTW radius {config.FASTDTW_RADIUS}.
- **{selection_label}:** {experiment['best_method']} dengan {int(best_row['Jumlah_Cluster'])} cluster, Silhouette {best_row['Silhouette_FastDTW']:.3f}, coverage {best_row['Coverage']:.1%}, dan η² minggu nol {best_row['Eta2_Minggu_Nol']:.3f}.
- **Metode tampilan awal:** {display_method} dengan {int(display_row['Jumlah_Cluster'])} cluster, Silhouette {display_row['Silhouette_FastDTW']:.3f}, dan coverage {display_row['Coverage']:.1%}.
- **Interpretasi bisnis:** profil produk harus kembali merujuk pada total penjualan, minggu nol, aktivitas, dan kontribusi volume dari data asli.

**Peringatan seleksi:**
{warning_lines if warning_lines else '- Tidak ada peringatan seleksi utama.'}

Skor multi-indeks adalah heuristik pemeringkatan internal, bukan bukti validitas eksternal. Jangan memakai label sebagai keputusan bisnis otomatis tanpa validasi domain dan uji sensitivitas.
"""
display(Markdown(conclusion_text))


### Ringkasan hasil eksperimen

- **Jumlah data:** 811 produk dan 52 minggu.
- **Preprocessing clustering:** Min-Max per produk dari nilai asli W0–W51.
- **Jarak pola:** FastDTW radius 2.
- **Pemimpin ranking setelah guardrail:** Agglomerative dengan 2 cluster, Silhouette 0.248, coverage 100.0%, dan η² minggu nol 0.019.
- **Metode tampilan awal:** KMedoids dengan 2 cluster, Silhouette 0.333, dan coverage 100.0%.
- **Interpretasi bisnis:** profil produk harus kembali merujuk pada total penjualan, minggu nol, aktivitas, dan kontribusi volume dari data asli.

**Peringatan seleksi:**
- Lolos guardrail eta-squared bukan bukti validitas eksternal; interpretasi bisnis tetap perlu divalidasi.

Skor multi-indeks adalah heuristik pemeringkatan internal, bukan bukti validitas eksternal. Jangan memakai label sebagai keputusan bisnis otomatis tanpa validasi domain dan uji sensitivitas.


## 14. Referensi metodologis

1. Salvador, S., & Chan, P. (2007). *Toward Accurate Dynamic Time Warping in Linear Time and Space*. Intelligent Data Analysis, 11(5), 561–580.
2. Petitjean, F., Ketterlin, A., & Gançarski, P. (2011). *A global averaging method for dynamic time warping, with applications to clustering*. Pattern Recognition, 44(3), 678–693.
3. Rousseeuw, P. J. (1987). *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis*. Journal of Computational and Applied Mathematics, 20, 53–65.
4. Davies, D. L., & Bouldin, D. W. (1979). *A Cluster Separation Measure*. IEEE Transactions on Pattern Analysis and Machine Intelligence, PAMI-1(2), 224–227.
5. Caliński, T., & Harabasz, J. (1974). *A dendrite method for cluster analysis*. Communications in Statistics, 3(1), 1–27.
6. Hubert, L., & Arabie, P. (1985). *Comparing partitions*. Journal of Classification, 2, 193–218.

---

**Catatan pengembangan berikutnya:** dashboard dapat dibuat dengan mengimpor fungsi dari `scripts/visualization.py` dan membaca tabel/model hasil dari `output/` serta `model/`. Dengan demikian, formula dan definisi chart tidak perlu dibuat ulang di `app.py`.